# Clase 12 — Caso de uso integrador: de datos crudos a un modelo predictivo

**Objetivos de la clase:**
- Ejecutar un pipeline completo de limpieza, análisis exploratorio y modelado de
  referencia sobre un dataset real.
- Relacionar cada paso del pipeline con las seis categorías de tareas de la Clase 1,
  cerrando el círculo del curso.

Basado en el capítulo 21 del manual y adaptado de `noaa-weather-data-v1.ipynb`
(corrigiendo 3 problemas de compatibilidad del notebook original: `np.bool`
eliminado de NumPy moderno, `fillna(method=...)` obsoleto en pandas moderno, y un
`!!pip install` con doble signo).

## El caso de uso

Datos meteorológicos horarios reales del aeropuerto JFK de Nueva York (114.546
observaciones, licencia CDLA-Sharing). Objetivo: ayudar a programar mejor los
horarios de vuelo anticipando la temperatura. Vamos a recorrer las tres fases del
capítulo 2: **limpieza**, **exploración (EDA)** y **modelado**.

In [ ]:
!pip install --quiet pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 15)


## Parte 1 — Limpieza de datos

In [ ]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fJKLoMKtgpDAl5MkA6rf2A/jfk-weather.csv"
crudo = pd.read_csv(url, parse_dates=["DATE"], low_memory=False)

# Nos quedamos con un subconjunto de columnas relevantes para el caso de uso
columnas = ["DATE", "HOURLYDRYBULBTEMPF", "HOURLYPrecip",
            "HOURLYWindSpeed", "HOURLYRelativeHumidity", "HOURLYStationPressure"]
clima = crudo[columnas].copy()
clima = clima.set_index(pd.DatetimeIndex(clima["DATE"])).drop(columns=["DATE"])

print(f"Filas antes de limpiar: {len(clima)}")
clima.head()


In [ ]:
# La columna de precipitación usa 'T' para "traza" (casi nada) y '*' para dato faltante
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].replace("T", "0.00")
clima["HOURLYPrecip"] = clima["HOURLYPrecip"].astype(str).str.replace("s", "", regex=False)

# El resto de columnas: '*' significa dato faltante
for columna in clima.columns:
    clima[columna] = pd.to_numeric(
        clima[columna].astype(str).str.replace("*", "", regex=False), errors="coerce"
    )

# Relleno de huecos hacia adelante (arreglo del "fillna(method=...)" obsoleto:
# se usa directamente .ffill())
clima = clima.ffill()
clima = clima.dropna()

print(f"Filas después de limpiar: {len(clima)}")
clima.describe()


## Parte 2 — Análisis exploratorio (EDA)

In [ ]:
# Tendencia de temperatura durante 2017
clima.loc["2017", "HOURLYDRYBULBTEMPF"].plot(figsize=(10, 4))
plt.title("Temperatura horaria en JFK durante 2017")
plt.ylabel("°F")
plt.show()


In [ ]:
# Mapa de correlación entre variables
# (arreglo del bug original: se usa dtype=bool en vez de np.bool, eliminado
# de NumPy moderno)
correlacion = clima.corr()
mascara = np.zeros_like(correlacion, dtype=bool)
mascara[np.triu_indices_from(mascara)] = True

plt.figure(figsize=(6, 5))
sns.heatmap(correlacion, mask=mascara, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlación entre variables meteorológicas")
plt.show()


## Parte 3 — Un primer modelo predictivo (baseline)

Antes de cualquier modelo sofisticado, se compara contra un modelo de referencia muy
simple: "la próxima hora tendrá la misma temperatura que la hora actual".

In [ ]:
from sklearn.metrics import mean_squared_error

temperatura = clima["HOURLYDRYBULBTEMPF"]

# Baseline: predecir "sin cambio" respecto a la hora anterior
prediccion_baseline = temperatura.shift(1).dropna()
real = temperatura.loc[prediccion_baseline.index]

rmse_baseline = np.sqrt(mean_squared_error(real, prediccion_baseline))
print(f"RMSE del modelo de referencia (sin cambio): {rmse_baseline:.2f} °F")


In [ ]:
# Un modelo apenas más elaborado: promedio de las últimas 3 horas
prediccion_media_movil = temperatura.rolling(3).mean().shift(1).dropna()
real_2 = temperatura.loc[prediccion_media_movil.index]

rmse_media_movil = np.sqrt(mean_squared_error(real_2, prediccion_media_movil))
print(f"RMSE del modelo de media móvil (3 horas): {rmse_media_movil:.2f} °F")

if rmse_media_movil < rmse_baseline:
    print("La media móvil mejora al modelo de referencia.")
else:
    print("La media móvil NO mejora al modelo de referencia.")


## Resumen y cierre del curso

- Recorriste el ciclo completo: gestión de datos -> limpieza (ETL) -> visualización
  (EDA) -> modelado -> evaluación — exactamente las seis tareas de la Clase 1.
- Un proyecto real casi nunca empieza por el modelo más sofisticado: empieza por
  limpiar los datos y comparar contra un modelo de referencia muy simple.
- Con esto termina el recorrido del curso. Todo lo que usaste aquí (Python, pandas,
  APIs, notebooks, Git/GitHub) es la misma caja de herramientas común a
  prácticamente cualquier proyecto real de datos.

Sigue con el notebook de práctica de esta clase para cerrar el curso con un
ejercicio propio sobre este mismo dataset.